# Simple Morphological Tree Examples

This notebook introduces the basic `mmcfilters` workflow on a small synthetic image and then repeats the same ideas on a real coin image.


## 1. Prepare the environment

Prepare the repository environment outside the notebook. Notebook-only dependencies are listed in `notebooks/requirements.txt`; install `mmcfilters` separately in the same environment. The next cell imports the active package directly and performs no local installation or build-tree loading.


In [1]:
import mmcfilters


## 2. Import the library


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2 as cv
import mmcfilters

def load_grayscale(path):
    image = cv.imread(str(path), cv.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(path)
    return np.ascontiguousarray(image, dtype=np.uint8)
from pathlib import Path
import mtviz as viz
if not hasattr(viz, "show_level_sets"):
    viz.show_level_sets = getattr(viz, "showLevelSets", lambda *args, **kwargs: None)
from bokeh.io import output_notebook, show
from bokeh.layouts import column, row
output_notebook()


def create_component_tree(image, is_maxtree, radius=1.5):
    if is_maxtree:
        return mmcfilters.MorphologicalTreeFactory.create_max_tree(image, radius=radius)
    return mmcfilters.MorphologicalTreeFactory.create_min_tree(image, radius=radius)


def print_tree_with_attribute(attribute_type, attribute_by_node):
    return lambda tree, node_id: f"id:{node_id}, {attribute_type.name}: {attribute_by_node[node_id]}"


def show_component_tree(tree, image=None, label=None):
    label = label or (lambda tree, node_id: f"id:{node_id}")

    def walk(node_id, depth=0):
        print("  " * depth + label(tree, node_id))
        for child_id in tree.children(node_id):
            walk(child_id, depth + 1)

    if image is not None:
        plt.figure(figsize=(5, 5))
        plt.imshow(image, cmap='gray', vmax=255, vmin=0)
        plt.axis('off')
        plt.show()

    walk(tree.root)


def show_tree(tree, label=None):
    show_component_tree(tree, label=label)


def getPlotTree(tree, title):
    show_component_tree(tree)
    return None


Loading BokehJS ...

## 3. Create a morphological tree from an input image

A component tree represents connected level sets as nodes. In this example, the synthetic image is small enough that the tree structure can be printed and compared with the pixel values.


In [3]:
#input_image = load_grayscale("../dat/imgTeste.png")

input_image = np.array([
        [203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203],
        [203,203,203, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78,203,203,203,203,203,203,203,203],
        [203,203, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78,203,203,203,203,203,203,203],
        [203,203, 78, 78,126,126,126,126,126,126,126, 78, 78, 78, 78, 78, 78, 78,203,203,203, 54, 54,203,203],
        [203,203, 78, 78,126, 38, 38, 38,126,126,126, 78, 78, 78, 78, 78, 78, 78,203,203, 54, 54, 54, 54,203],
        [203,203, 78, 78,126, 38, 38, 38,126,126,126, 78, 78, 78, 78, 78, 78, 78,203, 54, 54, 54, 80, 54,203],
        [203,203, 78, 78,126, 38, 38, 38,126, 78, 78, 78, 78, 78, 78, 78, 78, 78,203, 54, 80, 54, 54, 54,203],
        [203, 78, 78, 78,126, 38, 38,126,126, 78, 78, 78,203,203,203,203,203,203,203, 54, 54, 54, 54,203,203],
        [203, 78, 78, 78,126,126,126,126, 78, 78,203,203,203,203,203,203,203,203, 54, 54, 54, 54, 54,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78,203,203,253,253,253,203,203,203, 54, 80, 54, 54, 54,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78,203,203,253,253,253,203,203, 54, 54, 54, 54, 54,203,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78,203,203,253,253,253,203,203,203, 54, 54, 54,203,203,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78, 78,203,203,203,203,203,203,203,203,203,203,203,203,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78, 78,203,203,203,126,126,126,126,203,203,203,203,203,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78, 78,203,203,126,126,126,126,126,126,126,126,126,203,203,203],
        [203, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78,203,203,126,126,126,126,126,126, 72,126,126,203,203,203],
        [203, 78, 78, 78, 78, 78, 78,161,161,161, 78, 78,203,126,126,126,126,126,126, 72, 72,126,126,126,203],
        [203, 78, 78, 78, 78, 78,161,161,161,161,161, 78,203,126,126,126,126,126, 72, 72, 72,126,126,126,203],
        [203, 78, 78, 78, 78, 78,161, 30, 30, 30,161, 78,203,203,126,126,126, 72, 72, 72, 72, 72,126,126,203],
        [203, 78, 78, 78, 78, 78,161, 30, 90, 30,161, 78, 78,203,126,126, 72, 72, 72, 72, 72, 72, 72,126,203],
        [203, 78, 78, 78, 78, 78,161, 30, 30, 30,161, 78, 78,203,203,126,126,126,126,126,126,126,126,126,203],
        [203, 78, 78, 78, 78, 78,161,161,161,161,161, 78, 78, 78,203,203,126,126,126,126,126,126,126,203,203],
        [203, 78, 78, 78, 78, 78,161,161,161,161,161, 78, 78, 78,203,203,203,203,126,126,126,126,203,203,203],
        [203,203, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78,203,203,203,203,203,203,203,203,203,203],
        [203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203]
], dtype=np.uint8)
input_image = np.ascontiguousarray(input_image, dtype=np.uint8)
(num_rows, num_columns) = input_image.shape

is_max_tree = False
tree = create_component_tree(input_image, is_max_tree)
viz.show_level_sets(input_image)


In [4]:
show_component_tree(tree, image=input_image)

id:0
  id:1
    id:2
      id:3
        id:7
        id:10
      id:5
        id:11
    id:4
      id:8
    id:6
      id:9


/var/folders/3q/b07y7nwd1rg6xzdr9dk2wwzc0000gn/T/ipykernel_42390/4040404933.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Available attributes

Attributes summarize geometric, radiometric, or topological properties of each node. Listing the available attributes is a good first step before choosing a filtering decision rule.


In [5]:
attribute_enum = type(mmcfilters.Attribute.AREA)
describe = {
    attribute_name: mmcfilters.Attribute.describe(attribute_value)
    for attribute_name in dir(mmcfilters.Attribute)
    if attribute_name.isupper()
    for attribute_value in [getattr(mmcfilters.Attribute, attribute_name)]
    if isinstance(attribute_value, attribute_enum)
}

df = pd.DataFrame(describe.items(), columns=['Attribute type', 'Description'])
df.style.set_caption("<H3><b>Available attributes</b></H3>")

,Attribute type,Description
0,AREA,Area: Number of pixels in the connected component.
1,AVG_CHILD_HEIGHT_NODE,Average child height: Mean height of all direct child subtrees. Useful for measuring uniformity of the subtree structure.
2,AXIS_ORIENTATION,"Axis orientation: Angle of the principal inertia axis, computed from central moments. Indicates the dominant orientation of the shape."
3,BALANCE_NODE,Balance: Difference between the maximum and minimum heights among the subtrees of the children. Indicates branching symmetry.
4,BITQUAD_AREA,Bitquad area (Duda): Refined sub-pixel area estimation using fractional weights based on the geometric contribution of local 2x2 pixel patterns.
5,BITQUAD_CIRCULARITY,Bitquad circularity: Compactness measure defined as (4π x areaDuda) / perimeter². Values close to 1 indicate circular shapes; lower values suggest elongation or irregularity. Degenerate zero-perimeter supports return 0.
6,BITQUAD_LENGTH_AVERAGE,"Bitquad average length: Estimated average longitudinal extent per component, derived from the average perimeter with a zero fallback for non-positive Euler component estimates."
7,BITQUAD_NUMBER_EULER,"Bitquad Euler number: Topological invariant computed as the number of connected components minus the number of holes, using 2x2 pattern statistics under 4- or 8-connectivity."
8,BITQUAD_NUMBER_HOLES,"Bitquad number of holes: Number of interior holes in the component, derived from the Euler characteristic assuming a single connected object."
9,BITQUAD_PERIMETER,"Bitquad perimeter: Discrete approximation of the shape's boundary length, calculated by summing edge-contributing patterns in the 2x2 pixel grid."


## 5. Compute a single attribute

`compute_single_attribute` returns one value per tree node. The gray-height attribute is used here because it is easy to inspect on the printed tree.


In [6]:
attribute_type = mmcfilters.Attribute.GRAY_LEVEL_HEIGHT
attribute_by_node = mmcfilters.Attribute.compute_single_attribute(tree, attribute_type)

print(f"{attribute_type.name} (numpy): {attribute_by_node}")

GRAY_LEVEL_HEIGHT (numpy): [223. 173. 131.  88.  54.  60.  26.   0.   0.   0.   0.   0.]


In [7]:
show_tree(tree, print_tree_with_attribute(attribute_type, attribute_by_node))

id:0, GRAY_LEVEL_HEIGHT: 223.0
  id:1, GRAY_LEVEL_HEIGHT: 173.0
    id:2, GRAY_LEVEL_HEIGHT: 131.0
      id:3, GRAY_LEVEL_HEIGHT: 88.0
        id:7, GRAY_LEVEL_HEIGHT: 0.0
        id:10, GRAY_LEVEL_HEIGHT: 0.0
      id:5, GRAY_LEVEL_HEIGHT: 60.0
        id:11, GRAY_LEVEL_HEIGHT: 0.0
    id:4, GRAY_LEVEL_HEIGHT: 54.0
      id:8, GRAY_LEVEL_HEIGHT: 0.0
    id:6, GRAY_LEVEL_HEIGHT: 26.0
      id:9, GRAY_LEVEL_HEIGHT: 0.0


## 6. Compute multiple attributes

`compute_attributes` evaluates several attributes together and returns a node-by-attribute table. This is useful when a filter or analysis combines multiple decision rules.


In [8]:
attribute_indices, attribute_matrix = mmcfilters.Attribute.compute_attributes(tree, [mmcfilters.Attribute.AREA, mmcfilters.Attribute.GRAY_LEVEL_HEIGHT, mmcfilters.Attribute.VOLUME, mmcfilters.Attribute.RELATIVE_VOLUME])
#print(attribute_matrix[:, attribute_indices['GRAY_LEVEL_HEIGHT']])
pd.DataFrame(attribute_matrix, columns=attribute_indices, index=pd.Index(range(len(attribute_matrix)), name="NodeID"))

,AREA,VOLUME,RELATIVE_VOLUME,GRAY_LEVEL_HEIGHT
NodeID,,,,
0,625.0,82872.0,77409.0,223.0
1,616.0,80595.0,45984.0,173.0
2,277.0,24802.0,20535.0,131.0
3,244.0,20608.0,10582.0,88.0
4,84.0,9612.0,1074.0,54.0
5,9.0,330.0,497.0,60.0
6,38.0,2130.0,983.0,26.0
7,191.0,14898.0,191.0,0.0
8,18.0,1296.0,18.0,0.0


## 7. Attribute filtering

Attribute filters remove or preserve nodes according to a decision rule and then reconstruct an image from the modified tree. This example uses the subtractive rule with the gray-height values.


In [9]:
attribute_by_node = mmcfilters.Attribute.compute_single_attribute(tree, mmcfilters.Attribute.GRAY_LEVEL_HEIGHT)
threshold = 80

preservation_mask = mmcfilters.NodePreservationMask(attribute_by_node > threshold)
filtered_image = mmcfilters.SubtractiveAttributeFilter(tree).apply_subtractive_attribute_filter(preservation_mask)

plt.subplot(1,2, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,2, 2)
plt.imshow(filtered_image.reshape(num_rows, num_columns), cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter')

Text(0.5, 1.0, 'attribute filter')

## 8. Extract extinction values

Extinction values rank extrema by the importance of the attribute that disappears when components merge. They are often more stable than a fixed threshold on raw attribute values.


In [10]:
attribute_by_node = mmcfilters.Attribute.compute_single_attribute(tree, mmcfilters.Attribute.GRAY_LEVEL_HEIGHT) # the attribute must be increasing

In [11]:
extinction_values = mmcfilters.ExtinctionValues(tree, attribute_by_node)
for leaf_id, cutoff_node_id, extinction in extinction_values.get_regional_extrema():
    print("Regional extremum (leaf):", leaf_id)
    print("Extinction value: ", extinction)
    print("Persistence node where the regional extremum still exists:", cutoff_node_id)
    print("The regional extremum disappears at:", tree.parent(cutoff_node_id))
    print()

show_component_tree(tree, image=input_image)


Regional extremum (leaf): 7
Extinction value:  3.4028234663852886e+38
Persistence node where the regional extremum still exists: 0
The regional extremum disappears at: 0

Regional extremum (leaf): 11
Extinction value:  60.0
Persistence node where the regional extremum still exists: 5
The regional extremum disappears at: 2

Regional extremum (leaf): 8
Extinction value:  54.0
Persistence node where the regional extremum still exists: 4
The regional extremum disappears at: 1

Regional extremum (leaf): 9
Extinction value:  26.0
Persistence node where the regional extremum still exists: 6
The regional extremum disappears at: 1

Regional extremum (leaf): 10
Extinction value:  0.0
Persistence node where the regional extremum still exists: 10
The regional extremum disappears at: 3

id:0
  id:1
    id:2
      id:3
        id:7
        id:10
      id:5
        id:11
    id:4
      id:8
    id:6
      id:9


/var/folders/3q/b07y7nwd1rg6xzdr9dk2wwzc0000gn/T/ipykernel_42390/4040404933.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Filter by extinction values

Here the filter keeps only the most relevant extrema according to the extinction ranking. This provides a compact way to control the number of preserved structures.


In [12]:
attribute_by_node = mmcfilters.Attribute.compute_single_attribute(tree, mmcfilters.Attribute.GRAY_LEVEL_HEIGHT)
attribute_filter = mmcfilters.AttributeFilters(tree)

num_leaves_to_keep = 3 # keep num_leaves_to_keep leaves with the highest extinction values
selection_policy = mmcfilters.ExtinctionSelectionPolicy.by_top_k(num_leaves_to_keep)
filtered_image = attribute_filter.filtering_by_extinction(attribute_by_node, selection_policy)

plt.subplot(1,2, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,2, 2)
plt.imshow(filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('keeping 3 regional extrema')

Text(0.5, 1.0, 'keeping 3 regional extrema')

## 10. Work with the coin image

The same workflow is applied to a real image: direct attribute filtering, extinction-value filtering, and saliency-map construction.


In [13]:
# 1. Filtering

from skimage import data, img_as_float
input_image = np.ascontiguousarray(data.coins(), dtype=np.uint8)
(num_rows, num_columns) = input_image.shape

is_max_tree = True
tree = create_component_tree(input_image, is_max_tree)

attribute_by_node = mmcfilters.Attribute.compute_single_topology_attribute(tree, mmcfilters.Attribute.AREA)
threshold = 500

preservation_mask = mmcfilters.NodePreservationMask(attribute_by_node > threshold)
filtered_image = mmcfilters.SubtractiveAttributeFilter(tree).apply_subtractive_attribute_filter(preservation_mask)

plt.subplot(1,2, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,2, 2)
plt.imshow(filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter')

Text(0.5, 1.0, 'attribute filter')

In [14]:
# 2. Filtering by extinction values

input_image = np.ascontiguousarray(np.clip(filtered_image, 0, 255), dtype=np.uint8)

is_max_tree = True
tree = create_component_tree(input_image, is_max_tree)
attribute_filter = mmcfilters.AttributeFilters(tree)
attribute_by_node = mmcfilters.Attribute.compute_single_topology_attribute(tree, mmcfilters.Attribute.AREA)

num_leaves_to_keep = 6 # keep num_leaves_to_keep leaves with the highest extinction values
filtered_image_6 = attribute_filter.filtering_by_extinction(attribute_by_node, mmcfilters.ExtinctionSelectionPolicy.by_top_k(num_leaves_to_keep))

num_leaves_to_keep = 24 # keep num_leaves_to_keep leaves with the highest extinction values
filtered_image_24 = attribute_filter.filtering_by_extinction(attribute_by_node, mmcfilters.ExtinctionSelectionPolicy.by_top_k(num_leaves_to_keep))

plt.figure(figsize=(15, 5))
plt.subplot(1,3, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,3, 2)
plt.imshow(filtered_image_6, cmap='gray', vmax=255, vmin=0)
plt.title('keeping 6 regional extrema')

plt.subplot(1,3,3)
plt.imshow(filtered_image_24, cmap='gray', vmax=255, vmin=0)
plt.title('keeping 24 regional extrema')

Text(0.5, 1.0, 'keeping 24 regional extrema')

In [15]:

attribute_by_node = mmcfilters.Attribute.compute_single_topology_attribute(tree, mmcfilters.Attribute.AREA)
circularity = mmcfilters.Attribute.compute_single_topology_attribute(tree, mmcfilters.Attribute.CIRCULARITY)
extinction_values = mmcfilters.ExtinctionValues(tree, attribute_by_node) # the attribute must be increasing
num_leaves_to_keep = int(tree.num_leaf_nodes * 1) # 5% of the regional extrema
contours = mmcfilters.ContourComputation.extraction(tree)

contour_image = np.zeros((num_rows*num_columns), dtype=np.float32)
importance = num_leaves_to_keep
for leaf_id, cutoff_node_id, extinction in extinction_values.get_regional_extrema()[:num_leaves_to_keep]:
    for p in contours.get_contour(cutoff_node_id):
        contour_image[p]= circularity[cutoff_node_id]
        #contour_image[p]= importance
    importance -= 1


plt.figure(figsize=(5, 5))
plt.imshow(contour_image.reshape(num_rows, num_columns), cmap='Grays')
plt.axis('off')
plt.show()


/var/folders/3q/b07y7nwd1rg6xzdr9dk2wwzc0000gn/T/ipykernel_42390/1249998584.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Combine shape decision rules on contours

This final section uses area and circularity to build contour-based saliency maps. The approach is limited to increasing attributes, so the chosen decision rules should be checked before reuse.


In [16]:
attribute_by_node = mmcfilters.Attribute.compute_single_topology_attribute(tree, mmcfilters.Attribute.AREA)
circularity = mmcfilters.Attribute.compute_single_topology_attribute(tree, mmcfilters.Attribute.CIRCULARITY)
extinction_values = mmcfilters.ExtinctionValues(tree, attribute_by_node) # the attribute must be increasing
num_leaves_to_keep = int(tree.num_leaf_nodes * 1)

contours = mmcfilters.ContourComputation.extraction(tree)

contour_image = np.zeros((num_rows*num_columns), dtype=np.float32)
importance = num_leaves_to_keep
for leaf_id, cutoff_node_id, extinction in extinction_values.get_regional_extrema()[:num_leaves_to_keep]:
    for p in contours.get_contour(cutoff_node_id):
        contour_image[p]=circularity[cutoff_node_id]
    importance -= 1



plt.figure(figsize=(15, 5))
plt.subplot(1,2, 1)
plt.imshow(contour_image.reshape(num_rows, num_columns), cmap='Grays')
plt.axis('off')
plt.title('saliency map: importance is the circularity')

plt.subplot(1,2, 2)
saliency_map = extinction_values.contour_map(mmcfilters.ExtinctionSelectionPolicy.by_top_k(num_leaves_to_keep), mmcfilters.ExtinctionContourScorePolicy.RANK_SCORE)
plt.imshow(saliency_map, cmap='Grays')
plt.axis('off')
plt.title('saliency map: importance is the extinction values')
plt.show()


/var/folders/3q/b07y7nwd1rg6xzdr9dk2wwzc0000gn/T/ipykernel_42390/887521007.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
